### Working with data in PySpark

Here, the data cleaning and preparation means to, 
1. fix missing values
2. fix wrong data types
3. remove duplicates
4. handle outliers 
5. standardise formats
6. prepare data for model/analysis 

#### 1. Handling Missing Values

In [0]:
# dropping nulls
df.dropna()

In [0]:
# fill nulls
df.fillna()
df.fillna({"age":0, "salary":3000})

#### 2. Removing Duplicates:- 
same row appears multiple time

In [0]:
df.dropDuplicates() # for all
df.dropDupicates(["email"]) # for specific column 

#### 3. Fixing Data Types

(i) Fixing the datatypes, check first

In [0]:
df.printSchema()
# to hv an idea abt datatypes of all columns

(ii) fix of this 

In [0]:
df.withColumn("age", col("age").cast("int"))

#### 4. Renaming Columns

In [0]:
df.withColumnRenamed("Customer Name", "Customer_Name")

#### 5. Filtering Bad Data

- need to remove negative ages.
- need to remove impossible ages. 

In [0]:
df.filter(df.age>0)

#### 6. Handling Outliers

ex.- Salary = 1000000, (avg = 50000)

In [0]:
df.filter(df.salary < 1000000)

#### 7. Standardising Data

ex - "male", "Male", "M"

In [0]:
from pyspark.sql.functions import lower

In [0]:
df.withColumn("gender", lower(df.gender))

#### 8. Working with Data

ex - "25-03-2026", "03-25-2026"

In [0]:
df = df.withColumn("date", to_date("date", "dd-mm-yyyy"))

#### 9. Creating new columns 

In [0]:
from pyspark.sql.functions import col

In [0]:
df - df.withColumn("bonus", col("salary")*0.1)

#### 10.Dropping Unnecessary Columns

In [0]:
df.drop("Unwanted_Column")

#### Handling Missing Values

*1. Dropping Missing Values*

In [0]:
df.dropna()
df.dropna(how = "any")
df.dropna(how = "all")
dr.dropna(subset = ["age"])

*2. Filling Missing Values*

In [0]:
df.fillna(0)

In [0]:
df.fillna({
    "age" : 25,
    "salary":3000
})

**Using Mean/Median to fill**

In [0]:
from pyspark.sql.functions import mean

In [0]:
mean_value = df.select(mean("salary")).collect()[0][0]
df = df.fillna({"salary": mean_value})

*3. Filling Forward and Backward Fill(Time - Series)* 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import last

In [0]:
window = Window.orderBy("date").rowsBetween(-1, 0)
df = df.withColumn(
                    "filled value",
                    last("value", ignoreNulls = True)
                    .over(window)
)

*4. Conditional Filling*

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.withColumn(
                    "age",
                    when (col("age").isNull(), 25)
                    .otherwise(col("age"))
)

*5. Flag Missing Values*

In [0]:
from pyspark.sql.functions import col

In [0]:
df = df.withColumn(
                "salary_missing",
                col("salary").isNull()
)

*6. Replacing Specific Values*

In [0]:
df.replace("NA", None)

In [0]:
df.replace(" ", None)

### Data Normalisation and Transformations

- Normalisations:- scalling numbers - making values comparable. 

- Transformation:- changing structure/format - making data usable. 

#### DATA NORMALISATION

ex:- 
a. salary:- 10,000 - 1,00,000

b. age:- 18 - 60 

In this model,  will prioritize salary just coz it's bigger than others. \
this is technically wrong. 

**Ways to Normalize**

**1. Min - max scaling**

formula:- (x-min)/(max-min)

In [0]:
from pyspark.ml.features import MinMaxScaler, VectorAssembler
assembler = VectorAssembler(inputCols = ["Age"], outputCol = "age_vec")
df = assembler.transform(df)
Scaler = MinMaxScaler(inputCol = "age_vec", outputCol = "age_scaled")
df = Scaler.fit(df).transform(df)

**2. Standardisation (z-score)**

In [0]:
from pyspark.ml.feature import StandardScaler
scaler = StandardScaler(inputCol = "age_vec", outputCol = "age_scaled")
df = scaler.fit(df).transform(df)

#### DATA TRANSFORMATION

**1. Column Transformation**

In [0]:
from pyspark.sql.functions import col
df = df.withColumn("bonus", col("salary")*0.1)

2. 